# Типовые вопросы по истории атрибутов

Этот ноутбук разбирает типовые вопросы по изменению набора атрибутов источника во времени.

Идея одна и та же: если нужно понимать не только текущее состояние, но и историю, то обычных `INSERT` и `DELETE` недостаточно. Нужна модель с историей, то есть **SCD Type 2**.

Ниже будем использовать такие поля в таблице атрибутов источника:

- `source_id` — источник;
- `attribute_name` — имя атрибута;
- `start_date` — дата начала актуальности;
- `end_date` — дата конца актуальности;
- `is_current` — актуальна ли запись сейчас;
- `is_deleted` — логически удалён ли атрибут.


## 1. Сначала было 5 атрибутов, через месяц стало 10

### Ситуация
У источника был набор из 5 атрибутов. Через месяц пришли ещё 5 новых атрибутов. Старые 5 остаются актуальными.

### Что делать
Новые атрибуты нужно **добавить**, то есть для них создаются новые записи. Здесь действительно нужен `INSERT INTO`.

### Мини-пример
22 января у источника были: `attr_1 ... attr_5`.
22 февраля появились ещё: `attr_6 ... attr_10`.

Итог: в истории будет 10 активных записей, если старые атрибуты не отменялись.


In [ ]:
-- Кейс 1: добавились новые атрибуты
INSERT INTO dim_attributes_history (
    source_id,
    attribute_name,
    start_date,
    end_date,
    is_current,
    is_deleted
)
VALUES
    (1, 'attr_6', DATE '2024-02-22', DATE '9999-12-31', 1, 0),
    (1, 'attr_7', DATE '2024-02-22', DATE '9999-12-31', 1, 0),
    (1, 'attr_8', DATE '2024-02-22', DATE '9999-12-31', 1, 0),
    (1, 'attr_9', DATE '2024-02-22', DATE '9999-12-31', 1, 0),
    (1, 'attr_10', DATE '2024-02-22', DATE '9999-12-31', 1, 0);


## 2. Первые 3 атрибута деактивировались

### Ситуация
Раньше атрибуты были в источнике, а потом перестали быть актуальными.

### Что делать
**`DELETE` делать не нужно.** Если удалить строки физически, мы потеряем историю и не сможем потом ответить, что было раньше.

Правильный вариант: **закрыть актуальность** этих атрибутов. Для этого:

- ставим `end_date`;
- меняем `is_current = 0`;
- при необходимости ставим `is_deleted = 1`.

Это и есть историзация в духе `SCD Type 2`: запись не удаляется, а становится неактуальной.


In [ ]:
-- Кейс 2: атрибуты перестали быть актуальными
UPDATE dim_attributes_history
SET
    end_date = DATE '2024-03-21',
    is_current = 0,
    is_deleted = 1
WHERE source_id = 1
  AND attribute_name IN ('attr_1', 'attr_2', 'attr_3')
  AND is_current = 1;


## 3. Как понять, какая картина была 22 февраля

### Ситуация
Прошло несколько месяцев: часть атрибутов добавилась, часть исчезла. Нужно восстановить состояние на конкретную дату, например на **22 февраля**.

### Что делать
Нужно выбрать те записи, для которых интересующая дата попадает в интервал актуальности:

- `start_date <= нужная_дата`;
- `end_date >= нужная_дата`.

Если хранить историю правильно, такой срез можно получить обычным `SELECT`.

### Ключевая мысль
Чтобы ответить на вопрос про прошлое, данные надо было не удалять, а закрывать по дате окончания действия.


In [ ]:
-- Кейс 3: восстановить картину на 22 февраля
SELECT
    source_id,
    attribute_name,
    start_date,
    end_date,
    is_current,
    is_deleted
FROM dim_attributes_history
WHERE source_id = 1
  AND DATE '2024-02-22' BETWEEN start_date AND end_date
ORDER BY attribute_name;


## 4. Один источник: было 5 атрибутов, стало 10, потом 3 стали неактуальными

### Ситуация
Это полный жизненный цикл одного и того же источника:

1. сначала было 5 атрибутов;
2. потом стало 10;
3. потом 3 атрибута стали неактуальными.

### Что это за модель
Здесь уже недостаточно просто хранить текущий список атрибутов. Нужна таблица истории, где есть:

- `start_date`;
- `end_date`;
- `is_deleted`;
- `is_current`.

Это и есть подход **SCD Type 2** для справочника атрибутов.

### Как трактовать `is_deleted`
`is_deleted` не означает физическое удаление строки. Он означает, что атрибут логически выбыл из текущего состава источника. История при этом остаётся в таблице.

### Мини-пример состояния
- `attr_1`, `attr_2`, `attr_3`, `attr_4`, `attr_5` появились 22 января;
- `attr_6` ... `attr_10` появились 22 февраля;
- `attr_1`, `attr_2`, `attr_3` стали неактуальны 22 марта.

Тогда на 22 января активно 5 атрибутов, на 22 февраля активно 10, а после 22 марта активно 7.


In [ ]:
-- Кейс 4: рекомендуемая структура таблицы истории атрибутов
CREATE TABLE dim_attributes_history (
    source_id      INT,
    attribute_name VARCHAR(255),
    start_date     DATE,
    end_date       DATE,
    is_current     INT,
    is_deleted     INT
);

-- Текущее состояние источника на сегодня
SELECT
    source_id,
    attribute_name
FROM dim_attributes_history
WHERE is_current = 1
  AND is_deleted = 0
ORDER BY attribute_name;


## Короткие ответы по кейсам

1. Добавились новые атрибуты: `INSERT INTO` для новых записей.
2. Атрибуты деактивировались: не `DELETE`, а закрытие записи через `end_date`, `is_current = 0`, `is_deleted = 1`.
3. Нужна картина на дату в прошлом: запрос по интервалу `start_date` / `end_date`.
4. Для таких задач нужна историческая модель атрибутов с `start_date`, `end_date`, `is_current`, `is_deleted`, то есть `SCD Type 2`.
